# 01 - Exploracao STJ Metadados

Objetivo: explorar metadados da base de integras do STJ a partir dos arquivos `metadados*.json` salvos em subpastas por ano no Drive, sem carregar os textos grandes do ZIP.

Para ampliar o recorte temporal no futuro, basta baixar novos JSONs com `00_download_stj_metadados.ipynb`, ajustar `ANOS_ANALISE` e rerodar este notebook.


## 1. Ambiente

No Colab, clone o repositorio antes de abrir/rodar o notebook, ou abra o notebook diretamente pelo GitHub. Os dados brutos devem ficar no Google Drive, fora do Git.

In [55]:
# Rode esta celula no Colab se as dependencias ainda nao estiverem instaladas.
# !pip install -r requirements.txt

In [56]:
from pathlib import Path
import json
import re

import pandas as pd
from tqdm.auto import tqdm


In [57]:
from pathlib import Path
from google.colab import drive

MOUNT_POINT = Path("/content/drive")

if (MOUNT_POINT / "MyDrive").exists():
    print("Google Drive ja esta montado.")
else:
    drive.mount(str(MOUNT_POINT), force_remount=True)

DRIVE_DATA = Path("/content/drive/MyDrive/Mestrado/2026/llms/data")
METADATA_RAW = DRIVE_DATA / "raw" / "stj_integras_metadata"
REPORTS_DATA = DRIVE_DATA / "reports" / "summaries"

# Ajuste o recorte aqui. Use None para ler todos os anos disponiveis.
ANOS_ANALISE = [2026]

for path in [METADATA_RAW, REPORTS_DATA]:
    path.mkdir(parents=True, exist_ok=True)

print("METADATA_RAW:", METADATA_RAW)


Google Drive ja esta montado.
METADATA_RAW: /content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata


## 2. Inspecao dos arquivos de metadados

Esperado em `raw/stj_integras_metadata/` no Drive:

- subpastas por ano, como `2024/`, `2025/`, `2026/`;
- arquivos `metadados*.json` dentro de cada ano.

Este notebook nao precisa de ZIP nem CSV de dicionario para a EDA de metadados.


In [58]:
def metadata_sort_key(path: Path) -> str:
    match = re.fullmatch(r"metadados(\d{6}|\d{8})\.json", path.name)
    return match.group(1) if match else path.name

if ANOS_ANALISE is None:
    metadata_files = sorted(METADATA_RAW.rglob("metadados*.json"), key=metadata_sort_key)
else:
    metadata_files = []
    for ano in ANOS_ANALISE:
        year_dir = METADATA_RAW / str(ano)
        metadata_files.extend(year_dir.glob("metadados*.json"))
    metadata_files = sorted(metadata_files, key=metadata_sort_key)

if not metadata_files:
    raise FileNotFoundError(f"Nenhum arquivo metadados*.json encontrado para ANOS_ANALISE={ANOS_ANALISE} em {METADATA_RAW}")

print("ANOS_ANALISE:", ANOS_ANALISE if ANOS_ANALISE is not None else "todos")
print("Arquivos de metadados encontrados:", len(metadata_files))
print("Pastas:", sorted({p.parent.name for p in metadata_files}))
print("Primeiros:", [p.name for p in metadata_files[:5]])
print("Ultimos:", [p.name for p in metadata_files[-5:]])


ANOS_ANALISE: [2026]
Arquivos de metadados encontrados: 71
Pastas: ['2026']
Primeiros: ['metadados20260102.json', 'metadados20260105.json', 'metadados20260106.json', 'metadados20260107.json', 'metadados20260108.json']
Ultimos: ['metadados20260413.json', 'metadados20260414.json', 'metadados20260415.json', 'metadados20260416.json', 'metadados20260417.json']


In [59]:
def metadata_date_from_filename(filename: str) -> pd.Timestamp | None:
    match = re.fullmatch(r"metadados(\d{6}|\d{8})\.json", filename)
    if not match:
        return None
    raw_date = match.group(1)
    fmt = "%Y%m%d" if len(raw_date) == 8 else "%Y%m"
    return pd.to_datetime(raw_date, format=fmt, errors="coerce")


metadata_inventory = pd.DataFrame({"arquivo": [path.name for path in metadata_files]})
metadata_inventory["data_arquivo"] = metadata_inventory["arquivo"].map(metadata_date_from_filename)
metadata_inventory["ano_arquivo"] = metadata_inventory["data_arquivo"].dt.year
metadata_inventory["mes_arquivo"] = metadata_inventory["data_arquivo"].dt.to_period("M").astype(str)

print("Periodo dos arquivos:", metadata_inventory["data_arquivo"].min(), "a", metadata_inventory["data_arquivo"].max())
metadata_inventory.groupby("ano_arquivo").size().rename("arquivos").reset_index()


Periodo dos arquivos: 2026-01-02 00:00:00 a 2026-04-17 00:00:00


,ano_arquivo,arquivos
0,2026,71


## 3. Leitura combinada dos metadados


In [60]:
def load_metadata(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        df = pd.DataFrame(data)
    elif isinstance(data, dict):
        list_values = [v for v in data.values() if isinstance(v, list)]
        if len(list_values) == 1:
            df = pd.DataFrame(list_values[0])
        else:
            df = pd.json_normalize(data)
    else:
        raise ValueError(f"Formato JSON inesperado: {type(data)}")

    df["arquivo_origem"] = path.name
    df["data_arquivo_origem"] = metadata_date_from_filename(path.name)
    return df

metadata_dfs = []
failed_metadata_files = []

for path in tqdm(metadata_files):
    try:
        metadata_dfs.append(load_metadata(path))
    except Exception as exc:
        failed_metadata_files.append({"arquivo": path.name, "erro": repr(exc)})

if not metadata_dfs:
    raise RuntimeError("Nenhum arquivo de metadados foi lido com sucesso.")

metadata_df = pd.concat(metadata_dfs, ignore_index=True)

print("Arquivos lidos:", len(metadata_dfs))
print("Arquivos com erro:", len(failed_metadata_files))
print(f"Metadados combinados: {metadata_df.shape[0]:,} linhas x {metadata_df.shape[1]:,} colunas")

if failed_metadata_files:
    failed_path = REPORTS_DATA / "stj_metadata_read_failures.csv"
    pd.DataFrame(failed_metadata_files).to_csv(failed_path, index=False)
    print("Falhas salvas em:", failed_path)

metadata_df.head()


  0%|          | 0/71 [00:00<?, ?it/s]

Arquivos lidos: 71
Arquivos com erro: 0
Metadados combinados: 193,979 linhas x 14 colunas


,SeqDocumento,dataPublicacao,tipoDocumento,numeroRegistro,processo,dataRecebimento,dataDistribuição,NM_MINISTRO,recurso,teor,descricaoMonocratica,assuntos,arquivo_origem,data_arquivo_origem
0,353144180,2026-01-02,DECISÃO,202505108774,HC 1064230,2025-12-23,2025-12-29,HERMAN BENJAMIN,None,Negando,Denegado o Habeas Corpus de #{nome_da_parte} #...,12334,metadados20260102.json,2026-01-02
1,353145378,2026-01-02,DECISÃO,202505118087,HC 1064794,2025-12-28,2025-12-28,HERMAN BENJAMIN,None,Negando,Denegado o Habeas Corpus de #{nome_da_parte} #...,14227,metadados20260102.json,2026-01-02
2,353132129,2026-01-02,DECISÃO,202505107537,HC 1063952,2025-12-22,2025-12-26,HERMAN BENJAMIN,None,Negando,Denegado o Habeas Corpus de #{nome_da_parte} #...,3548,metadados20260102.json,2026-01-02
3,353144178,2026-01-02,DECISÃO,202505111903,HC 1064410,2025-12-24,2025-12-26,HERMAN BENJAMIN,None,Negando,Denegado o Habeas Corpus de #{nome_da_parte} #...,4355,metadados20260102.json,2026-01-02
4,353132787,2026-01-02,DECISÃO,202505101918,HC 1063889,2025-12-20,2025-12-29,HERMAN BENJAMIN,None,Negando,Denegado o Habeas Corpus de #{nome_da_parte} #...,4355,metadados20260102.json,2026-01-02


In [61]:
metadata_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193979 entries, 0 to 193978
Data columns (total 14 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   SeqDocumento          193979 non-null  int64         
 1   dataPublicacao        193979 non-null  object        
 2   tipoDocumento         193979 non-null  object        
 3   numeroRegistro        193979 non-null  object        
 4   processo              193979 non-null  object        
 5   dataRecebimento       193979 non-null  object        
 6   dataDistribuição      193974 non-null  object        
 7   NM_MINISTRO           193979 non-null  object        
 8   recurso               54222 non-null   object        
 9   teor                  193979 non-null  object        
 10  descricaoMonocratica  135650 non-null  object        
 11  assuntos              193870 non-null  object        
 12  arquivo_origem        193979 non-null  object        
 13 

In [62]:
metadata_df.columns.tolist()

['SeqDocumento',
 'dataPublicacao',
 'tipoDocumento',
 'numeroRegistro',
 'processo',
 'dataRecebimento',
 'dataDistribuição',
 'NM_MINISTRO',
 'recurso',
 'teor',
 'descricaoMonocratica',
 'assuntos',
 'arquivo_origem',
 'data_arquivo_origem']

## 4. Analise exploratoria dos metadados

Antes de analisar texto integral, esta etapa descreve o universo de documentos e processos disponivel nos metadados: volume por ano, tipo de documento, ministro, teor, recurso e assuntos.

In [63]:
def parse_stj_date_series(series: pd.Series) -> pd.Series:
    text = series.astype("string").str.strip()
    parsed = pd.to_datetime(text, errors="coerce")
    missing = parsed.isna() & text.notna()
    if missing.any():
        parsed.loc[missing] = pd.to_datetime(text.loc[missing], format="%Y%m%d", errors="coerce")
    return parsed


def find_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    normalized = {col.casefold(): col for col in df.columns}
    for candidate in candidates:
        if candidate.casefold() in normalized:
            return normalized[candidate.casefold()]
    return None

DATE_COLUMNS = {
    "dataPublicacao": ["dataPublicacao"],
    "dataRecebimento": ["dataRecebimento"],
    "dataDistribuicao": ["dataDistribuicao", "dataDistribuição"],
}

MINISTER_COLUMN = find_column(metadata_df, ["ministro", "NM_MINISTRO"])
PROCESS_COLUMN = find_column(metadata_df, ["processo"])
REGISTRATION_COLUMN = find_column(metadata_df, ["numeroRegistro"])

metadata_eda = metadata_df.copy()

for canonical_name, candidates in DATE_COLUMNS.items():
    source_col = find_column(metadata_eda, candidates)
    if source_col:
        metadata_eda[f"{canonical_name}_dt"] = parse_stj_date_series(metadata_eda[source_col])
        metadata_eda[f"ano_{canonical_name}"] = metadata_eda[f"{canonical_name}_dt"].dt.year

if MINISTER_COLUMN and "ministro" not in metadata_eda.columns:
    metadata_eda["ministro"] = metadata_eda[MINISTER_COLUMN]

print("Coluna de ministro:", MINISTER_COLUMN)
print("Coluna de processo:", PROCESS_COLUMN)
print("Coluna de registro:", REGISTRATION_COLUMN)
metadata_eda.head()

Coluna de ministro: NM_MINISTRO
Coluna de processo: processo
Coluna de registro: numeroRegistro


,SeqDocumento,dataPublicacao,tipoDocumento,numeroRegistro,processo,dataRecebimento,dataDistribuição,NM_MINISTRO,recurso,teor,...,assuntos,arquivo_origem,data_arquivo_origem,dataPublicacao_dt,ano_dataPublicacao,dataRecebimento_dt,ano_dataRecebimento,dataDistribuicao_dt,ano_dataDistribuicao,ministro
0,353144180,2026-01-02,DECISÃO,202505108774,HC 1064230,2025-12-23,2025-12-29,HERMAN BENJAMIN,None,Negando,...,12334,metadados20260102.json,2026-01-02,2026-01-02,2026,2025-12-23,2025,2025-12-29,2025.0,HERMAN BENJAMIN
1,353145378,2026-01-02,DECISÃO,202505118087,HC 1064794,2025-12-28,2025-12-28,HERMAN BENJAMIN,None,Negando,...,14227,metadados20260102.json,2026-01-02,2026-01-02,2026,2025-12-28,2025,2025-12-28,2025.0,HERMAN BENJAMIN
2,353132129,2026-01-02,DECISÃO,202505107537,HC 1063952,2025-12-22,2025-12-26,HERMAN BENJAMIN,None,Negando,...,3548,metadados20260102.json,2026-01-02,2026-01-02,2026,2025-12-22,2025,2025-12-26,2025.0,HERMAN BENJAMIN
3,353144178,2026-01-02,DECISÃO,202505111903,HC 1064410,2025-12-24,2025-12-26,HERMAN BENJAMIN,None,Negando,...,4355,metadados20260102.json,2026-01-02,2026-01-02,2026,2025-12-24,2025,2025-12-26,2025.0,HERMAN BENJAMIN
4,353132787,2026-01-02,DECISÃO,202505101918,HC 1063889,2025-12-20,2025-12-29,HERMAN BENJAMIN,None,Negando,...,4355,metadados20260102.json,2026-01-02,2026-01-02,2026,2025-12-20,2025,2025-12-29,2025.0,HERMAN BENJAMIN


In [64]:
def count_docs_by(df: pd.DataFrame, column: str, n: int | None = None) -> pd.DataFrame:
    if column not in df.columns:
        return pd.DataFrame(columns=[column, "documentos"])
    counts = (
        df[column]
        .fillna("(vazio)")
        .astype(str)
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="documentos")
    )
    return counts.head(n) if n else counts

summary = {
    "arquivos_origem": metadata_eda["arquivo_origem"].nunique() if "arquivo_origem" in metadata_eda.columns else None,
    "documentos": len(metadata_eda),
    "processos_unicos": metadata_eda[PROCESS_COLUMN].nunique() if PROCESS_COLUMN else None,
    "registros_unicos": metadata_eda[REGISTRATION_COLUMN].nunique() if REGISTRATION_COLUMN else None,
    "seqdocumento_unicos": metadata_eda["SeqDocumento"].nunique() if "SeqDocumento" in metadata_eda.columns else None,
    "data_publicacao_min": metadata_eda["dataPublicacao_dt"].min() if "dataPublicacao_dt" in metadata_eda.columns else None,
    "data_publicacao_max": metadata_eda["dataPublicacao_dt"].max() if "dataPublicacao_dt" in metadata_eda.columns else None,
}

summary

{'arquivos_origem': 71,
 'documentos': 193979,
 'processos_unicos': 180938,
 'registros_unicos': 180634,
 'seqdocumento_unicos': 193884,
 'data_publicacao_min': Timestamp('2026-01-02 00:00:00'),
 'data_publicacao_max': Timestamp('2026-04-17 00:00:00')}

In [65]:
docs_by_publication_year = count_docs_by(metadata_eda, "ano_dataPublicacao")
docs_by_publication_year

,ano_dataPublicacao,documentos
0,2026,193979


In [66]:
if "dataPublicacao_dt" in metadata_eda.columns:
    metadata_eda["mes_dataPublicacao"] = metadata_eda["dataPublicacao_dt"].dt.to_period("M").astype(str)
    docs_by_publication_month = count_docs_by(metadata_eda, "mes_dataPublicacao")
else:
    docs_by_publication_month = pd.DataFrame(columns=["mes_dataPublicacao", "documentos"])

docs_by_publication_month


,mes_dataPublicacao,documentos
0,2026-03,89527
1,2026-02,58274
2,2026-04,33598
3,2026-01,12580


In [67]:
if PROCESS_COLUMN and "ano_dataPublicacao" in metadata_eda.columns:
    processes_by_publication_year = (
        metadata_eda
        .dropna(subset=["ano_dataPublicacao"])
        .groupby("ano_dataPublicacao")[PROCESS_COLUMN]
        .nunique()
        .reset_index(name="processos_unicos")
        .sort_values("ano_dataPublicacao")
    )
else:
    processes_by_publication_year = pd.DataFrame(columns=["ano_dataPublicacao", "processos_unicos"])

processes_by_publication_year

,ano_dataPublicacao,processos_unicos
0,2026,180938


In [68]:
docs_by_type = count_docs_by(metadata_eda, "tipoDocumento")
docs_by_type.head(30)

,tipoDocumento,documentos
0,DECISÃO,142211
1,ACÓRDÃO,51768


In [69]:
if "ano_dataPublicacao" in metadata_eda.columns and "tipoDocumento" in metadata_eda.columns:
    docs_by_year_type = pd.crosstab(
        metadata_eda["ano_dataPublicacao"],
        metadata_eda["tipoDocumento"],
        margins=True,
    )
else:
    docs_by_year_type = pd.DataFrame()

docs_by_year_type

tipoDocumento,ACÓRDÃO,DECISÃO,All
ano_dataPublicacao,,,
2026,51768,142211,193979
All,51768,142211,193979


In [70]:
docs_by_minister = count_docs_by(metadata_eda, "ministro")
docs_by_minister.head(30)

,ministro,documentos
0,HERMAN BENJAMIN,52529
1,HUMBERTO MARTINS,7558
2,JOÃO OTÁVIO DE NORONHA,7014
3,MARIA ISABEL GALLOTTI,6356
4,PAULO SÉRGIO DOMINGUES,6103
5,LUIS FELIPE SALOMÃO,5790
6,DANIELA TEIXEIRA,5520
7,RAUL ARAÚJO,5504
8,REYNALDO SOARES DA FONSECA,5157
9,SEBASTIÃO REIS JÚNIOR,5080


In [71]:
docs_by_teor = count_docs_by(metadata_eda, "teor")
docs_by_teor.head(30)

,teor,documentos
0,Não Conhecendo,86577
1,Negando,79390
2,Concedendo,15160
3,Outros,12330
4,Admitindo Embargo,522


In [72]:
docs_by_recurso = count_docs_by(metadata_eda, "recurso")
docs_by_recurso.head(30)

,recurso,documentos
0,(vazio),139757
1,AgInt,21961
2,EDcl,14879
3,AgRg,11571
4,RE,2716
5,DESIS,1283
6,PET,799
7,Acordo,452
8,RCD,181
9,PExt,91


In [73]:
TOP_N_ASSUNTOS = 20

# O campo "assuntos" vem como codigo/trilha hierarquica, nao como rotulo textual.
# Exemplo: "00899.07681.09580.09607.". Nesta etapa contamos os codigos
# sem tentar inferir nomes; o mapeamento para descricoes deve vir de uma tabela
# propria de assuntos/classes, se ela for incorporada depois.
def normalize_assunto_code(code: str) -> str:
    text = str(code).strip()
    if not text:
        return ""
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    if re.fullmatch(r"\d+", text):
        return text.zfill(5)
    return text


def normalize_assunto_path(path: str) -> str:
    raw_parts = [part for part in str(path).strip().split(".") if part]
    parts = [normalize_assunto_code(part) for part in raw_parts]
    parts = [part for part in parts if part]
    if not parts:
        return ""
    return ".".join(parts) + "."


def split_assunto_paths(value) -> list[str]:
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    parts = re.split(r"\s*[;|,]\s*", text)
    normalized_parts = [normalize_assunto_path(part) for part in parts]
    return [part for part in normalized_parts if part]


def assunto_levels(path: str) -> dict[str, str | None]:
    codes = [part for part in str(path).split(".") if part]
    return {
        "assunto_raiz": codes[0] if codes else None,
        "assunto_final": codes[-1] if codes else None,
        "assunto_profundidade": len(codes),
    }

if "assuntos" in metadata_eda.columns:
    assuntos_long = (
        metadata_eda[["SeqDocumento", "ano_dataPublicacao", "assuntos"]]
        .assign(assunto_path=lambda df: df["assuntos"].map(split_assunto_paths))
        .explode("assunto_path")
        .dropna(subset=["assunto_path"])
    )
    assunto_levels_df = pd.DataFrame(
        assuntos_long["assunto_path"].map(assunto_levels).tolist(),
        index=assuntos_long.index,
    )
    assuntos_long = pd.concat(
        [assuntos_long.drop(columns=["assuntos"]), assunto_levels_df],
        axis=1,
    )
else:
    assuntos_long = pd.DataFrame(
        columns=["SeqDocumento", "ano_dataPublicacao", "assunto_path", "assunto_raiz", "assunto_final", "assunto_profundidade"]
    )

docs_by_assunto_path = (
    assuntos_long["assunto_path"]
    .value_counts()
    .rename_axis("assunto_path")
    .reset_index(name="ocorrencias")
)
docs_by_assunto_root = (
    assuntos_long["assunto_raiz"]
    .value_counts()
    .rename_axis("assunto_raiz")
    .reset_index(name="ocorrencias")
)
docs_by_assunto_final = (
    assuntos_long["assunto_final"]
    .value_counts()
    .rename_axis("assunto_final")
    .reset_index(name="ocorrencias")
)

top_assunto_paths = docs_by_assunto_path.head(TOP_N_ASSUNTOS)["assunto_path"].tolist()
top_assunto_finais = docs_by_assunto_final.head(TOP_N_ASSUNTOS)["assunto_final"].tolist()

assuntos_top_path = (
    assuntos_long.loc[assuntos_long["assunto_path"].isin(top_assunto_paths), ["ano_dataPublicacao", "assunto_path"]]
    .reset_index(drop=True)
)
assuntos_top_final = (
    assuntos_long.loc[assuntos_long["assunto_final"].isin(top_assunto_finais), ["ano_dataPublicacao", "assunto_final"]]
    .reset_index(drop=True)
)

docs_by_year_top_assunto_path = pd.crosstab(
    assuntos_top_path["ano_dataPublicacao"],
    assuntos_top_path["assunto_path"],
)

docs_by_year_top_assunto_final = pd.crosstab(
    assuntos_top_final["ano_dataPublicacao"],
    assuntos_top_final["assunto_final"],
)

docs_by_assunto_final.head(30)


,assunto_final,ocorrencias
0,03608,19885
1,03372,7124
2,10433,5118
3,04355,5054
4,05897,4991
5,05566,4485
6,03633,4306
7,10439,3881
8,09596,3838
9,09587,3755


In [74]:
# Enriquecimento opcional dos assuntos com a tabela CNJ/STJ parseada no notebook 04.
# Se o arquivo nao existir, a EDA continua funcionando apenas com os codigos.
ASSUNTOS_LOOKUP_CANDIDATES = [
    Path("data/reference/assuntos/processed/assuntos_lookup.parquet"),
    Path("data/reference/assuntos/processed/assuntos_lookup.csv"),
    DRIVE_DATA / "reference" / "assuntos" / "processed" / "assuntos_lookup.parquet",
    DRIVE_DATA / "reference" / "assuntos" / "processed" / "assuntos_lookup.csv",
]

ASSUNTOS_LOOKUP_PATH = next((path for path in ASSUNTOS_LOOKUP_CANDIDATES if path.exists()), None)

if ASSUNTOS_LOOKUP_PATH is None:
    assuntos_lookup = pd.DataFrame()
    print("Tabela de assuntos nao encontrada. Rode o notebook 04 para gerar o lookup.")
elif ASSUNTOS_LOOKUP_PATH.suffix == ".parquet":
    assuntos_lookup = pd.read_parquet(ASSUNTOS_LOOKUP_PATH)
    print("Tabela de assuntos carregada:", ASSUNTOS_LOOKUP_PATH)
else:
    assuntos_lookup = pd.read_csv(ASSUNTOS_LOOKUP_PATH, dtype={"codigo": "string", "codigo_pai": "string"})
    print("Tabela de assuntos carregada:", ASSUNTOS_LOOKUP_PATH)

if not assuntos_lookup.empty:
    assuntos_lookup = assuntos_lookup.copy()
    assuntos_lookup["codigo"] = assuntos_lookup["codigo"].map(normalize_assunto_code)
    if "codigo_pai" in assuntos_lookup.columns:
        assuntos_lookup["codigo_pai"] = assuntos_lookup["codigo_pai"].map(
            lambda value: normalize_assunto_code(value) if pd.notna(value) else value
        )
    lookup_cols = [
        col for col in ["codigo", "assunto", "codigo_pai", "caminho_assuntos", "instancia", "fonte"]
        if col in assuntos_lookup.columns
    ]
    assuntos_lookup = assuntos_lookup[lookup_cols].drop_duplicates(subset=["codigo"])

def label_assunto_path(path: str) -> str:
    if assuntos_lookup.empty:
        return path
    labels = assuntos_lookup.set_index("codigo")["assunto"].to_dict()
    codes = [part for part in str(path).split(".") if part]
    return " > ".join(labels.get(code, code) for code in codes)

def enrich_assunto_counts(table: pd.DataFrame, code_column: str) -> pd.DataFrame:
    if assuntos_lookup.empty or table.empty:
        return table.copy()
    return table.merge(
        assuntos_lookup,
        how="left",
        left_on=code_column,
        right_on="codigo",
    ).drop(columns=["codigo"])

docs_by_assunto_final_labeled = enrich_assunto_counts(docs_by_assunto_final, "assunto_final")
docs_by_assunto_root_labeled = enrich_assunto_counts(docs_by_assunto_root, "assunto_raiz")
docs_by_assunto_path_labeled = docs_by_assunto_path.copy()
docs_by_assunto_path_labeled["assunto_path_rotulo"] = docs_by_assunto_path_labeled["assunto_path"].map(label_assunto_path)

lookup_coverage = {
    "lookup_path": str(ASSUNTOS_LOOKUP_PATH) if ASSUNTOS_LOOKUP_PATH else None,
    "codigos_finais_com_rotulo": int(docs_by_assunto_final_labeled.get("assunto", pd.Series(dtype="object")).notna().sum()) if not assuntos_lookup.empty else 0,
    "codigos_finais_total": int(len(docs_by_assunto_final)),
    "codigos_raiz_com_rotulo": int(docs_by_assunto_root_labeled.get("assunto", pd.Series(dtype="object")).notna().sum()) if not assuntos_lookup.empty else 0,
    "codigos_raiz_total": int(len(docs_by_assunto_root)),
}
lookup_coverage


Tabela de assuntos nao encontrada. Rode o notebook 04 para gerar o lookup.


{'lookup_path': None,
 'codigos_finais_com_rotulo': 0,
 'codigos_finais_total': 2571,
 'codigos_raiz_com_rotulo': 0,
 'codigos_raiz_total': 1220}

In [75]:
metadata_reports = {
    "stj_docs_by_publication_year.csv": docs_by_publication_year,
    "stj_docs_by_publication_month.csv": docs_by_publication_month,
    "stj_processes_by_publication_year.csv": processes_by_publication_year,
    "stj_docs_by_type.csv": docs_by_type,
    "stj_docs_by_minister.csv": docs_by_minister,
    "stj_docs_by_teor.csv": docs_by_teor,
    "stj_docs_by_recurso.csv": docs_by_recurso,
    "stj_docs_by_assunto_path.csv": docs_by_assunto_path,
    "stj_docs_by_assunto_root.csv": docs_by_assunto_root,
    "stj_docs_by_assunto_final.csv": docs_by_assunto_final,
    "stj_docs_by_assunto_path_labeled.csv": docs_by_assunto_path_labeled,
    "stj_docs_by_assunto_root_labeled.csv": docs_by_assunto_root_labeled,
    "stj_docs_by_assunto_final_labeled.csv": docs_by_assunto_final_labeled,
    "stj_docs_by_year_top_assunto_path.csv": docs_by_year_top_assunto_path.reset_index(),
    "stj_docs_by_year_top_assunto_final.csv": docs_by_year_top_assunto_final.reset_index(),
}

if not docs_by_year_type.empty:
    metadata_reports["stj_docs_by_year_type.csv"] = docs_by_year_type.reset_index()

for filename, table in metadata_reports.items():
    output_path = REPORTS_DATA / filename
    table.to_csv(output_path, index=False)
    print("Salvo:", output_path)


Salvo: /content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/stj_docs_by_publication_year.csv
Salvo: /content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/stj_docs_by_publication_month.csv
Salvo: /content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/stj_processes_by_publication_year.csv
Salvo: /content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/stj_docs_by_type.csv
Salvo: /content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/stj_docs_by_minister.csv
Salvo: /content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/stj_docs_by_teor.csv
Salvo: /content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/stj_docs_by_recurso.csv
Salvo: /content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/stj_docs_by_assunto_path.csv
Salvo: /content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/stj_docs_by_assunto_root.csv
Salvo: /content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/stj_docs_by_assunto_final.

In [76]:
lookup_status = (
    f"Tabela de assuntos carregada de `{ASSUNTOS_LOOKUP_PATH}`."
    if ASSUNTOS_LOOKUP_PATH
    else "Tabela de assuntos ainda nao carregada; os codigos aparecem sem rotulo textual."
)

final_subject_cols = [
    col for col in ["assunto_final", "assunto", "caminho_assuntos", "ocorrencias"]
    if col in docs_by_assunto_final_labeled.columns
]
root_subject_cols = [
    col for col in ["assunto_raiz", "assunto", "caminho_assuntos", "ocorrencias"]
    if col in docs_by_assunto_root_labeled.columns
]
path_subject_cols = ["assunto_path", "assunto_path_rotulo", "ocorrencias"]

metadata_summary_md = f"""# EDA de metadados - STJ Integras

## Totais

- Arquivos de origem: {summary['arquivos_origem']:,}
- Documentos: {summary['documentos']:,}
- Processos unicos: {summary['processos_unicos']:,}
- Registros unicos: {summary['registros_unicos']:,}
- SeqDocumento unicos: {summary['seqdocumento_unicos']:,}
- Periodo de publicacao: {summary['data_publicacao_min']} a {summary['data_publicacao_max']}

## Documentos por ano de publicacao

{docs_by_publication_year.to_markdown(index=False)}

## Documentos por mes de publicacao

{docs_by_publication_month.to_markdown(index=False)}

## Processos unicos por ano de publicacao

{processes_by_publication_year.to_markdown(index=False)}

## Tipos de documento mais frequentes

{docs_by_type.head(20).to_markdown(index=False)}

## Ministros mais frequentes

{docs_by_minister.head(20).to_markdown(index=False)}

## Teor

{docs_by_teor.head(20).to_markdown(index=False)}

## Recursos

{docs_by_recurso.head(20).to_markdown(index=False)}

## Assuntos

O campo `assuntos` foi tratado como codigo/trilha hierarquica. Os codigos foram normalizados como strings de 5 digitos por nivel. {lookup_status}

Cobertura do lookup: {lookup_coverage}

### Codigos finais de assunto mais frequentes

{docs_by_assunto_final_labeled[final_subject_cols].head(20).to_markdown(index=False)}

### Trilhas completas de assunto mais frequentes

{docs_by_assunto_path_labeled[path_subject_cols].head(20).to_markdown(index=False)}

### Codigos raiz de assunto mais frequentes

{docs_by_assunto_root_labeled[root_subject_cols].head(20).to_markdown(index=False)}

## Observacao sobre cruzamentos ano a ano

Contagens por ano para documentos e processos sao pequenas e legiveis. Para assuntos, o cruzamento completo pode ficar muito esparso; por isso o notebook salva apenas os Top {TOP_N_ASSUNTOS} assuntos por caminho completo e por codigo final.
"""

metadata_summary_path = REPORTS_DATA / "metadata_eda_summary.md"
metadata_summary_path.write_text(metadata_summary_md, encoding="utf-8")
metadata_summary_path


PosixPath('/content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/metadata_eda_summary.md')

## 5. Proximos passos

- Revisar os CSVs e `metadata_eda_summary.md` em `reports/summaries`.
- Se o recorte fizer sentido, rodar `02_validacao_integras_txt.ipynb` para validar a ligacao com os arquivos TXT.
- Manter a analise textual para uma etapa posterior, com amostra controlada.
